# Prompt Engineering

### What is `Groq`?

Groq is a prominent AI technology company known for developing a specialized microchip called the Language Processing Unit (LPU). It is designed to run AI models—such as LLaMA or Mixtral—with near-instant, real-time response speeds and at a very low cost.

**The Core Innovation: LPU vs. GPU**

Unlike traditional Graphics Processing Units (GPUs) that are often constrained by complex memory caching and data bottlenecks, Groq's custom LPU is built specifically for AI inference.

Speed: LPUs process AI responses at unprecedented speeds, significantly reducing the waiting time you might experience with other standard AI systems.

Determinism: At every tick of its clock, the system knows exactly where data is and where it is going, eliminating greedy, overlapping calculations.


**Installation required (in your virtual environment):**

- groq library for API calling: `pip install groq`

- getpass library (for password / API key management): `pip install getpass4`

In [2]:
import os
from groq import Groq
from getpass import getpass

In [3]:
if os.environ.get('GROQ_API_KEY') is None:
    os.environ['GROQ_API_KEY'] = getpass("GROQ_API_KEY")

In [4]:
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

In [5]:
# Function to call Groq API

def get_response(prompt):

    completion = client.chat.completions.create(
      model="llama-3.3-70b-versatile",
      messages=[
          {
              "role": "user",
              "content": prompt,
          },
      ],
    )

    return completion.choices[0].message.content

## Zero Shot Classification

In [7]:
complaint = (
    "The delivery guy just left the package at the gate without ringing the bell. "
    "I found it two hours later completely soaked from the rain."
)

# Without Chain of Thought Prompt
basic_prompt = f"""
Classify the following customer complaint into one of these categories:
1. Delivery issue
2. Product quality
3. Customer service
4. Other

Complaint: "{complaint}"

Category:"""

print("Basic Prompt:\n", basic_prompt, "\n")

Basic Prompt:
 
Classify the following customer complaint into one of these categories:
1. Delivery issue
2. Product quality
3. Customer service
4. Other

Complaint: "The delivery guy just left the package at the gate without ringing the bell. I found it two hours later completely soaked from the rain."

Category: 



In [9]:
print(get_response(basic_prompt))

Category: 1. Delivery issue. 

The complaint is specifically about the delivery process, as the package was left unattended and exposed to the elements, rather than being handed over to the customer directly.


## Few Shots Classification

In [10]:
# Few-shot examples
few_shot_prompt = f"""
Classify the following customer complaints into one of these categories:
1. Delivery issue
2. Product quality
3. Customer service
4. Other

Examples:

Complaint: "My order was supposed to arrive yesterday, but it hasn’t shown up yet."
Category: Delivery issue

Complaint: "The shoes I received had a tear on the side."
Category: Product quality

Complaint: "The support agent was rude and didn’t help at all."
Category: Customer service

Complaint: "I had trouble applying the discount code during checkout."
Category: Other

Now classify the following complaint:

Complaint: {complaint}
Category:"""

In [11]:
# Complaint to classify
complaint_to_classify = (
    "The delivery guy just left the package at the gate without ringing the bell. "
    "I found it two hours later completely soaked from the rain."
)

In [12]:
print(few_shot_prompt.format({"complaint": complaint_to_classify[0]}))


Classify the following customer complaints into one of these categories:
1. Delivery issue
2. Product quality
3. Customer service
4. Other

Examples:

Complaint: "My order was supposed to arrive yesterday, but it hasn’t shown up yet."
Category: Delivery issue

Complaint: "The shoes I received had a tear on the side."
Category: Product quality

Complaint: "The support agent was rude and didn’t help at all."
Category: Customer service

Complaint: "I had trouble applying the discount code during checkout."
Category: Other

Now classify the following complaint:

Complaint: The delivery guy just left the package at the gate without ringing the bell. I found it two hours later completely soaked from the rain.
Category:


In [13]:
print(get_response(few_shot_prompt.format({"complaint": complaint_to_classify[0]})))

Category: Delivery issue

The complaint is related to the delivery process, specifically how the package was left at the customer's location, which led to it getting damaged. This falls under the category of a delivery issue.


## Chain of Thought (CoT)

In [15]:
# Function to call OpenAI
def get_response_v1(system_prompt, prompt):

    print( "**************** Prompt *****************")
    print(system_prompt + prompt)
    print( "****************************************")

    completion = client.chat.completions.create(
      model="llama-3.3-70b-versatile",
      messages=[
          {
              "role": "system",
              "content": system_prompt
        },
          {
              "role": "user",
              "content": prompt,
          },
      ],
    )

    return completion.choices[0].message.content

In [16]:
system_prompt = """You are a helpful assistant and your job is to classify complaints into one of the following categories:

Delivery issue
Product quality
Customer service
Other

Use chain-of-thought reasoning to guide the classification of the following complaint."""

user_prompt = f"""
Complaint: {complaint}
"""

print("With Chain of Thought:")
print(get_response_v1(system_prompt,
                      user_prompt.format({"complaint":
                                         "The delivery guy just left the package at the gate without ringing the bell. I found it two hours later completely soaked from the rain."})))

With Chain of Thought:
**************** Prompt *****************
You are a helpful assistant and your job is to classify complaints into one of the following categories:

Delivery issue
Product quality
Customer service
Other

Use chain-of-thought reasoning to guide the classification of the following complaint.
Complaint: The delivery guy just left the package at the gate without ringing the bell. I found it two hours later completely soaked from the rain.

****************************************
To classify this complaint, let's break it down step by step:

1. The complaint mentions a "delivery guy" and a "package," which immediately suggests that the issue is related to the delivery process.
2. The specific problem stated is that the package was left at the gate without the delivery person notifying the customer (by ringing the bell), which led to the package being exposed to the rain for an extended period.
3. The consequence of this action (or lack thereof) is that the package bec

## Structured Response

In [17]:
# System prompt
system_prompt = "You are a helpful assistant that classifies customer complaints. Return the result only as a JSON object: {\"category\": \"...\"}"

In [18]:
user_prompt = f"""
Classify the customer complaint into one of these categories:
1. Delivery issue
2. Product quality
3. Customer service
4. Other

Complaint: {complaint}
"""

In [19]:
resp = get_response_v1(system_prompt,
                      user_prompt.format({"complaint":
                                         "The delivery guy just left the package at the gate without ringing the bell. I found it two hours later completely soaked from the rain."}))

**************** Prompt *****************
You are a helpful assistant that classifies customer complaints. Return the result only as a JSON object: {"category": "..."}
Classify the customer complaint into one of these categories:
1. Delivery issue
2. Product quality
3. Customer service
4. Other

Complaint: The delivery guy just left the package at the gate without ringing the bell. I found it two hours later completely soaked from the rain.

****************************************


In [20]:
resp

'{"category": "Delivery issue"}'

In [21]:
import json

category_dict = json.loads(resp)

print(category_dict)
print(category_dict["category"])

{'category': 'Delivery issue'}
Delivery issue


## Temperature

In [22]:
# API wrapper
def get_response_v2(system_prompt, prompt, temperature=0.7):

    print( "**************** Prompt *****************")
    print(system_prompt + prompt)
    print( "****************************************")

    completion = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
        temperature=temperature
    )
    return completion.choices[0].message.content.strip()

In [23]:
system_prompt = """Your job is to classify complaints into one of the following categories:

Delivery issue
Product quality
Customer service
Other

Use chain-of-thought reasoning to guide the classification of the following complaint."""

user_prompt = f"""
Complaint: {complaint}
"""

In [24]:
print(get_response_v2(system_prompt,
                      user_prompt.format({"complaint":
                                         "The delivery guy just left the package at the gate without ringing the bell. I found it two hours later completely soaked from the rain."})))

**************** Prompt *****************
Your job is to classify complaints into one of the following categories:

Delivery issue
Product quality
Customer service
Other

Use chain-of-thought reasoning to guide the classification of the following complaint.
Complaint: The delivery guy just left the package at the gate without ringing the bell. I found it two hours later completely soaked from the rain.

****************************************
To classify this complaint, let's break it down step by step:

1. The issue revolves around a package that was left at the gate.
2. The package was not handed over to the customer directly, and the delivery person did not notify the customer of its arrival (by ringing the bell).
3. As a result, the package was exposed to the rain for two hours, causing damage (it became soaked).

Considering these points, the complaint is primarily about the way the package was handled during delivery. The customer is unhappy with how the delivery was executed, w

In [27]:
print(get_response_v2(system_prompt,
                      user_prompt.format({"complaint":
                                         "The delivery guy just left the package at the gate without ringing the bell. I found it two hours later completely soaked from the rain."}),
                      temperature = 2.0))

**************** Prompt *****************
Your job is to classify complaints into one of the following categories:

Delivery issue
Product quality
Customer service
Other

Use chain-of-thought reasoning to guide the classification of the following complaint.
Complaint: The delivery guy just left the package at the gate without ringing the bell. I found it two hours later completely soaked from the rain.

****************************************
To classify this complaint, let's break it down step by step:

1. The complaint is about a package being left at the gate, which indicates that the issue is related to how the package was handled during delivery.
2. The package was not handed over to the customer directly, and the customer was not aware of its arrival until later.
3. The customer found the package soaked from the rain, which suggests that the package was left unattended and exposed to the elements.
4. The customer's concern seems to be with the manner in which the delivery was ca

## Self Consistency

Self-consistency in prompt engineering is an advanced decoding strategy that improves an AI's accuracy by generating multiple reasoning paths for a single prompt and selecting the final answer via majority voting. It acts as a safety net against the single-path failures of standard Chain-of-Thought (CoT) prompting.

In [28]:
import time
from collections import Counter

system_prompt = "You are a helpful assistant that classifies customer complaints. Return the result only as a JSON object: {\"category\": \"...\"}"

user_prompt = f"""
Classify the customer complaint into one of these categories:
1. Delivery issue
2. Product quality
3. Customer service
4. Other

Use chain-of-thought reasoning to guide the classification of the following complaint.

Complaint: {complaint}
"""

complaint = "I waited all day at home, but the package never came. When I contacted support,  they told me to wait another 48 hours without giving a reason."

print(user_prompt.format({"complaint": complaint}))

results = []

for temp in [0.0, 0.5, 2.0]:
    try:
        resp = get_response_v2(system_prompt
                              , user_prompt.format({"complaint": complaint})
                              , temperature = temp)
        category_dict = json.loads(resp)
        category = category_dict['category']
        if category is not None:
            results.append(category)
    except Exception as e:
        print(f"Error on run {temp}: {e}")

    time.sleep(1)  # to avoid rate limits

    counts = Counter(results)
    majority = counts.most_common(1)[0]

print("All predictions:", results)
print("Majority vote:", majority)


Classify the customer complaint into one of these categories:
1. Delivery issue
2. Product quality
3. Customer service
4. Other

Use chain-of-thought reasoning to guide the classification of the following complaint.

Complaint: The delivery guy just left the package at the gate without ringing the bell. I found it two hours later completely soaked from the rain.

**************** Prompt *****************
You are a helpful assistant that classifies customer complaints. Return the result only as a JSON object: {"category": "..."}
Classify the customer complaint into one of these categories:
1. Delivery issue
2. Product quality
3. Customer service
4. Other

Use chain-of-thought reasoning to guide the classification of the following complaint.

Complaint: The delivery guy just left the package at the gate without ringing the bell. I found it two hours later completely soaked from the rain.

****************************************
**************** Prompt *****************
You are a helpfu